# Computation of timings in GVA dataset

only applicable to the patients received IV-thrombolysis or endovascular therapy. 

Timings: 
- ODT( Onset to Door Time): from disease onset (last known well time) to the time of arriving hospital
- ONT (Onset to Needle Time): time from disease onset (last known well time) to the time of receiving IV-thrombolysis
- DNT (Door to Needle Time): time from arriving at hospital to the time of receiving IV-thrombolysis
- DPT (Door to Puncture TIme): time from arriving at hospital to the time of puncture (EVT)
- OPT (Onset to Puncture Time)

In [ ]:
import pandas as pd

In [ ]:
registry_path = '/Users/jk/stroke_datasets/stroke_registry_post_hoc_modified.xlsx'

In [ ]:
registry_df = pd.read_excel(registry_path)

Arrival at hospital
Arrival time

Onset date
Onset time

IVT start date
IVT start time

Onset to treatment (min.)
Door to treatment (min.)

Date of groin puncture
Time of groin puncture

Onset to groin puncture (min.)
Door to groin puncture (min.)


In [ ]:
registry_df 

In [ ]:
# onset datetime


In [ ]:
# convert to datetime

def _normalize_time(t):
    if pd.isna(t):
        return '00:00'
    s = str(t).strip()
    if ':' in s:
        parts = s.split(':')
        hh = parts[0].zfill(2)
        mm = parts[1].zfill(2) if len(parts) > 1 and parts[1].isdigit() else '00'
        return f"{hh}:{mm}"
    digits = ''.join(ch for ch in s if ch.isdigit())
    if len(digits) == 3:
        digits = '0' + digits
    if len(digits) == 4:
        return digits[:2] + ':' + digits[2:]
    return '00:00'

def _normalize_date(d):
    if pd.isna(d):
        return None
    s = str(int(d)) if isinstance(d, (int, float)) and not pd.isna(d) else str(d).strip()
    digits = ''.join(ch for ch in s if ch.isdigit())
    if len(digits) == 8:
        return digits  # YYYYMMDD
    try:
        parsed = pd.to_datetime(s, dayfirst=False)
        return parsed.strftime('%Y%m%d')
    except Exception:
        try:
            parsed = pd.to_datetime(s, dayfirst=True)
            return parsed.strftime('%Y%m%d')
        except Exception:
            return None

def _parse_date_time(date_val, time_val):
    date_norm = _normalize_date(date_val)
    if not date_norm:
        return pd.NaT
    time_norm = _normalize_time(time_val)
    try:
        return pd.to_datetime(date_norm + ' ' + time_norm, format='%Y%m%d %H:%M')
    except Exception:
        try:
            return pd.to_datetime(date_norm + time_norm.replace(':', ''), format='%Y%m%d%H%M')
        except Exception:
            try:
                return pd.to_datetime(date_norm, format='%Y%m%d')
            except Exception:
                return pd.NaT




In [ ]:
registry_df['onset_datetime'] = registry_df.apply(lambda r: _parse_date_time(r.get('Onset date'), r.get('Onset time')), axis=1)
registry_df['admission_datetime'] = registry_df.apply(lambda r: _parse_date_time(r.get('Arrival at hospital'), r.get('Arrival time')), axis=1)
registry_df['IVT_datetime'] = registry_df.apply(lambda r: _parse_date_time(r.get('IVT start date'), r.get('IVT start time')), axis=1)
registry_df['IAT_datetime'] = registry_df.apply(lambda r: _parse_date_time(r.get('Date of groin puncture'), r.get('Time of groin puncture')), axis=1)

In [ ]:
registry_df[['Onset date', 'Onset time', 'onset_datetime', 'Arrival at hospital', 'Arrival time', 'admission_datetime', 'IVT start date', 'IVT start time', 'IVT_datetime', 'Date of groin puncture', 'Time of groin puncture', 'IAT_datetime']]

In [ ]:
# - ODT( Onset to Door Time): from disease onset (last known well time) to the time of arriving hospital
# - ONT (Onset to Needle Time): time from disease onset (last known well time) to the time of receiving IV-thrombolysis
# - DNT (Door to Needle Time): time from arriving at hospital to the time of receiving IV-thrombolysis
# - DPT (Door to Puncture TIme): time from arriving at hospital to the time of puncture (EVT)

registry_df['ODT'] = (registry_df['admission_datetime'] - registry_df['onset_datetime']).dt.total_seconds() / 60
registry_df['ONT'] = (registry_df['IVT_datetime'] - registry_df['onset_datetime']).dt.total_seconds() / 60
# replace na by Onset to treatment (min.)
registry_df['ONT'].fillna(registry_df['Onset to treatment (min.)'], inplace=True)
registry_df['DNT'] = (registry_df['IVT_datetime'] - registry_df['admission_datetime']).dt.total_seconds() / 60
# replace na by Door to treatment (min.)
registry_df['DNT'].fillna(registry_df['Door to treatment (min.)'], inplace=True)
registry_df['DPT'] = (registry_df['IAT_datetime'] - registry_df['admission_datetime']).dt.total_seconds() / 60
registry_df['DPT'].fillna(registry_df['Door to groin puncture (min.)'], inplace=True)
registry_df['OPT'] = (registry_df['IAT_datetime'] - registry_df['onset_datetime']).dt.total_seconds() / 60
registry_df['OPT'].fillna(registry_df['Onset to groin puncture (min.)'], inplace=True)

In [ ]:
print(registry_df['ODT'].describe(), '\n')
print(registry_df['ONT'].describe(), '\n')
print(registry_df['DNT'].describe(), '\n')
print(registry_df['DPT'].describe(), '\n')